In [ ]:
"""
biomining.py — Biomedical NER / Entity Linking / Relation Extraction at scale
============================================================================
Staged pipeline for ~300k OpenAlex abstracts in Google Colab. Every stage
writes TIDY parquet shards and is resumable — a Colab disconnect costs you at
most one shard.

STAGES (run in this order; each is independent and skippable)
------------------------------------------------------------
  Stage 0  recover_pmids()        DOI -> PMID via NCBI ID Converter (shrinks the
                                  no-PMID set BEFORE you pay for inference)
           fetch_pubtator_bulk()  download PubTator3 bulk TSVs (once)
           join_pubtator()        entities + relations for PMID records, FREE
                                  (no inference: genes, diseases, chemicals,
                                   species, cell lines, variants, 12 relations)
  Stage 1  mesh_from_openalex()   turn the oa_mesh you ALREADY have into tidy
                                  disease/chemical entities — zero cost
  Stage 2  run_scispacy()         CPU spans + MeSH/RxNorm/GO/HPO linking
           run_hunflair2()        GPU NER + normalization (best accuracy)
           run_gliner()           GPU zero-shot NER with YOUR labels
                                  (tick species, vector, host, acaricide, ...)
  Stage 3  llm_batch_build()      JSONL for OpenAI/Anthropic batch APIs
           llm_batch_parse()      -> relations + pathways / dosages / endpoints /
                                     adverse events / biomarkers / MoA
                                  (the categories NO off-the-shelf tool covers)
  Stage 4  crosswalk()            MeSH->MONDO, NCBI Gene->HGNC, etc.
           combine()              one entities table + one relations table

TIDY SCHEMAS (everything joins back on oa_id)
---------------------------------------------
  entities : oa_id, source, entity_type, mention, concept_id, ontology, score
  relations: oa_id, source, subj_id, subj_type, predicate, obj_id, obj_type, evidence

Colab quickstart
----------------
    from google.colab import drive; drive.mount('/content/drive')
    !pip -q install duckdb pandas pyarrow requests
    import biomining as bm
    BASE = "/content/drive/MyDrive/ticks/bio"
    df = pd.read_parquet(".../canonical_screened.parquet")
    df = df[df.oa_relevance_tier.isin(["relevant","screen"])]      # screen first!

    df   = bm.recover_pmids(df, base=BASE)                          # Stage 0
    bm.fetch_pubtator_bulk(base=BASE)                               #  (~6 GB, once)
    bm.join_pubtator(df, base=BASE)
    bm.mesh_from_openalex(df, base=BASE)                            # Stage 1
    todo = bm.no_pubtator_subset(df, base=BASE)                     # the ~200k
    bm.run_gliner(todo, base=BASE)                                  # Stage 2 (GPU)
    bm.llm_batch_build(todo, base=BASE)                             # Stage 3
    ents, rels = bm.combine(base=BASE)                              # Stage 4
"""

from __future__ import annotations
import os, re, io, json, time, glob, gzip, math
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------------- schemas ---
ENTITY_COLS   = ["oa_id","source","entity_type","mention","concept_id","ontology","score"]
RELATION_COLS = ["oa_id","source","subj_id","subj_type","predicate","obj_id","obj_type","evidence"]

# PubTator3 bulk files (https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/)
PUBTATOR_FTP = "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/"
PUBTATOR_FILES = {   # name -> (filename, approx size)
    "gene":     ("gene2pubtator3.gz",      "716 MB"),
    "disease":  ("disease2pubtator3.gz",   "2.0 GB"),
    "chemical": ("chemical2pubtator3.gz",  "1.7 GB"),
    "species":  ("species2pubtator3.gz",   "461 MB"),
    "cellline": ("cellline2pubtator3.gz",  "68 MB"),
    "mutation": ("mutation2pubtator3.gz",  "110 MB"),
    "relation": ("relation2pubtator3.gz",  "277 MB"),
}
# entity_type -> the ontology PubTator3 normalizes to
PUBTATOR_ONTOLOGY = {
    "gene": "NCBI Gene", "disease": "MeSH", "chemical": "MeSH",
    "species": "NCBI Taxonomy", "cellline": "Cellosaurus", "mutation": "dbSNP/HGVS",
}

# Custom labels for GLiNER — the domain-specific stuff fixed-schema tools miss.
TICK_LABELS = [
    "tick species", "tick genus", "pathogen", "disease", "vector", "host animal",
    "reservoir host", "acaricide", "chemical or drug", "gene or protein",
    "biomarker", "geographic location", "control method", "diagnostic method",
]


def _d(base, *p):
    q = Path(base).joinpath(*p); q.mkdir(parents=True, exist_ok=True); return q


# ============================================================ shard runner ===
class ShardRunner:
    """Run `fn(shard_df) -> tidy DataFrame` over a corpus in resumable shards.

    Writes one parquet per shard into <base>/<stage>/. On re-run, shards already
    on disk are skipped — so a Colab disconnect costs you at most one shard.
    """

    def __init__(self, base, stage, shard_size=5000, id_col="oa_id"):
        self.dir = _d(base, stage)
        self.shard_size = shard_size
        self.id_col = id_col
        self.stage = stage

    def done_shards(self):
        return {int(re.search(r"shard_(\d+)", p).group(1))
                for p in glob.glob(str(self.dir / "shard_*.parquet"))}

    def run(self, df, fn, verbose=True):
        n = len(df)
        n_shards = math.ceil(n / self.shard_size)
        done = self.done_shards()
        if verbose:
            print(f"[{self.stage}] {n:,} rows -> {n_shards} shards "
                  f"({len(done)} already done)")
        for i in range(n_shards):
            if i in done:
                continue
            part = df.iloc[i * self.shard_size:(i + 1) * self.shard_size]
            t0 = time.time()
            out = fn(part)
            if out is None or len(out) == 0:
                out = pd.DataFrame(columns=ENTITY_COLS)
            out.to_parquet(self.dir / f"shard_{i:05d}.parquet", index=False)
            if verbose:
                rate = len(part) / max(time.time() - t0, 1e-9)
                eta = (n_shards - i - 1) * self.shard_size / max(rate, 1e-9) / 60
                print(f"  shard {i+1}/{n_shards}  {len(out):,} rows  "
                      f"{rate:.1f} docs/s  ETA {eta:.0f} min")
        return self.load()

    def load(self):
        parts = sorted(glob.glob(str(self.dir / "shard_*.parquet")))
        if not parts:
            return pd.DataFrame(columns=ENTITY_COLS)
        return pd.concat((pd.read_parquet(p) for p in parts), ignore_index=True)


# =================================================== STAGE 0: PMID + PubTator =
def recover_pmids(df, base, id_col="oa_id", batch=200, sleep=0.4):
    """DOI -> PMID via NCBI ID Converter. Run this FIRST: every PMID you recover
    is one abstract you do NOT have to run inference on (PubTator3 has it free).
    Adds/fills `oa_pmid`. Cached to <base>/pmid_map.parquet."""
    import requests
    cache = Path(base) / "pmid_map.parquet"
    known = pd.read_parquet(cache) if cache.exists() else pd.DataFrame(columns=["oa_doi", "pmid"])

    have = df["oa_pmid"].astype(str).str.strip().replace({"nan": "", "None": ""})
    need = df.loc[have.eq("") & df["oa_doi"].astype(str).str.startswith("10."), "oa_doi"]
    need = need[~need.isin(known["oa_doi"])].dropna().unique().tolist()
    print(f"[stage0] have PMID: {(~have.eq('')).sum():,} | resolving {len(need):,} DOIs")

    rows = []
    url = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    for i in range(0, len(need), batch):
        chunk = need[i:i + batch]
        try:
            r = requests.get(url, params={"ids": ",".join(chunk), "format": "json",
                                          "tool": "biomining", "email": "user@example.org"},
                             timeout=60)
            for rec in r.json().get("records", []):
                if rec.get("pmid"):
                    rows.append({"oa_doi": rec.get("doi", ""), "pmid": rec["pmid"]})
        except Exception as e:
            print(f"  chunk {i}: {e}")
        time.sleep(sleep)
        if (i // batch) % 25 == 0 and rows:
            pd.concat([known, pd.DataFrame(rows)]).drop_duplicates("oa_doi").to_parquet(cache, index=False)

    known = pd.concat([known, pd.DataFrame(rows)]).drop_duplicates("oa_doi")
    known.to_parquet(cache, index=False)

    out = df.merge(known, on="oa_doi", how="left")
    out["oa_pmid"] = out["oa_pmid"].astype(str).str.strip().replace({"nan": "", "None": ""})
    out.loc[out["oa_pmid"].eq(""), "oa_pmid"] = out.loc[out["oa_pmid"].eq(""), "pmid"]
    out = out.drop(columns=["pmid"])
    got = out["oa_pmid"].astype(str).ne("").sum()
    print(f"[stage0] PMID coverage now {got:,}/{len(out):,} ({got/len(out):.1%})")
    return out


def fetch_pubtator_bulk(base, which=("disease", "chemical", "gene", "species",
                                     "cellline", "mutation", "relation")):
    """Download PubTator3 bulk TSVs (~5-6 GB total). One-time; cached on Drive."""
    import requests
    d = _d(base, "pubtator_bulk")
    for key in which:
        fn, size = PUBTATOR_FILES[key]
        dest = d / fn
        if dest.exists():
            print(f"  {fn} already present"); continue
        print(f"  downloading {fn} (~{size}) ...")
        with requests.get(PUBTATOR_FTP + fn, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(dest, "wb") as f:
                for c in r.iter_content(chunk_size=1 << 20):
                    f.write(c)
        print(f"    done: {dest.stat().st_size/1e6:.0f} MB")
    return d


def join_pubtator(df, base, which=("disease", "chemical", "gene", "species",
                                   "cellline", "mutation", "relation")):
    """Stream the bulk TSVs with DuckDB, keep only OUR PMIDs, emit tidy
    entities + relations. This is the free lunch: no inference at all."""
    import duckdb
    bulk = Path(base) / "pubtator_bulk"
    out = _d(base, "stage0_pubtator")

    pm = df[["oa_id", "oa_pmid"]].copy()
    pm["oa_pmid"] = pm["oa_pmid"].astype(str).str.strip()
    pm = pm[pm["oa_pmid"].ne("") & pm["oa_pmid"].str.isdigit()]
    if not len(pm):
        print("[stage0] no PMIDs — skipping"); return None, None
    print(f"[stage0] joining PubTator3 for {len(pm):,} PMID records")

    con = duckdb.connect()
    con.register("pm", pm)
    ents, rels = [], []
    for key in which:
        f = bulk / PUBTATOR_FILES[key][0]
        if not f.exists():
            print(f"  (missing {f.name} — skip)"); continue
        if key == "relation":
            # relation2pubtator3: PMID, relation_type, concept_1, concept_2
            q = f"""
                SELECT p.oa_id, 'pubtator3' AS source,
                       CAST(t.column1 AS VARCHAR) AS predicate,
                       CAST(t.column2 AS VARCHAR) AS subj_id,
                       CAST(t.column3 AS VARCHAR) AS obj_id
                FROM read_csv('{f}', delim='\t', header=false, quote='',
                              ignore_errors=true, all_varchar=true) t
                JOIN pm p ON p.oa_pmid = CAST(t.column0 AS VARCHAR)
            """
            r = con.execute(q).df()
            if len(r):
                # relation file encodes concepts as "Type|ID" (e.g. "Species|6945")
                sp = r["subj_id"].str.split("|", n=1)
                op = r["obj_id"].str.split("|", n=1)
                r["subj_type"] = sp.str[0]
                r["subj_id"] = sp.str[-1]
                r["obj_type"] = op.str[0]
                r["obj_id"] = op.str[-1]
                r["evidence"] = ""
                rels.append(r[RELATION_COLS])
        else:
            # <type>2pubtator3: PMID, Type, ConceptID, Mentions, Resource
            q = f"""
                SELECT p.oa_id, 'pubtator3' AS source,
                       lower(CAST(t.column1 AS VARCHAR)) AS entity_type,
                       CAST(t.column3 AS VARCHAR) AS mention,
                       CAST(t.column2 AS VARCHAR) AS concept_id
                FROM read_csv('{f}', delim='\t', header=false, quote='',
                              ignore_errors=true, all_varchar=true) t
                JOIN pm p ON p.oa_pmid = CAST(t.column0 AS VARCHAR)
            """
            e = con.execute(q).df()
            if len(e):
                e["ontology"] = PUBTATOR_ONTOLOGY.get(key, "")
                e["score"] = 1.0
                ents.append(e[ENTITY_COLS])
        print(f"  {key}: {len(ents[-1]) if (key!='relation' and ents) else (len(rels[-1]) if rels and key=='relation' else 0):,} rows")

    E = pd.concat(ents, ignore_index=True) if ents else pd.DataFrame(columns=ENTITY_COLS)
    R = pd.concat(rels, ignore_index=True) if rels else pd.DataFrame(columns=RELATION_COLS)
    E.to_parquet(out / "entities.parquet", index=False)
    R.to_parquet(out / "relations.parquet", index=False)
    print(f"[stage0] {len(E):,} entities | {len(R):,} relations "
          f"| {E.oa_id.nunique():,} docs covered")
    return E, R


def no_pubtator_subset(df, base):
    """The records PubTator3 could NOT cover -> these are what you run NER on."""
    p = Path(base) / "stage0_pubtator" / "entities.parquet"
    covered = set(pd.read_parquet(p)["oa_id"]) if p.exists() else set()
    todo = df[~df["oa_id"].isin(covered)].copy()
    print(f"[stage2] PubTator covered {len(covered):,}; "
          f"{len(todo):,} records need local NER")
    return todo


# ==================================================== STAGE 1: free MeSH ======
def mesh_from_openalex(df, base, sep=" | "):
    """You ALREADY have MeSH for ~1/3 of records (oa_mesh). Turn it into tidy
    entities for free — this substitutes for disease/chemical NER on those docs."""
    out = _d(base, "stage1_mesh")
    s = df[["oa_id", "oa_mesh"]].copy()
    s["oa_mesh"] = s["oa_mesh"].astype("string").fillna("")
    s = s[s["oa_mesh"].ne("")]
    s["mention"] = s["oa_mesh"].str.split(re.escape(sep).replace("\\ ", " "))
    e = s.explode("mention")
    e["mention"] = e["mention"].str.strip()
    e = e[e["mention"].ne("")]
    e["source"] = "openalex_mesh"
    e["entity_type"] = "mesh_descriptor"
    e["concept_id"] = ""          # OpenAlex gives descriptor names, not UIs
    e["ontology"] = "MeSH"
    e["score"] = 1.0
    e = e[ENTITY_COLS]
    e.to_parquet(out / "entities.parquet", index=False)
    print(f"[stage1] {len(e):,} MeSH entities from {e.oa_id.nunique():,} docs (free)")
    return e


# ================================================ STAGE 2: local NER (GPU) ====
def _text(row):
    t = str(row.get("oa_title") or "")
    a = str(row.get("oa_abstract") or "")
    k = str(row.get("oa_abstract") or "")
    return (t + ". " + a + ". " + k).strip()


def run_gliner(df, base, labels=None, model="Ihor/gliner-biomed-bi-large-v1.0",
               shard_size=5000, batch_size=16, threshold=0.5, device=None):
    """Zero-shot NER with YOUR OWN labels — the only tool that captures
    vector/host/reservoir/acaricide roles that fixed-schema taggers miss.
    NOTE: GLiNER gives typed SPANS, not ontology IDs. Normalize in stage 4.
        !pip install gliner
    """
    from gliner import GLiNER
    import torch
    labels = labels or TICK_LABELS
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[gliner] {model} on {device} | {len(labels)} labels")
    m = GLiNER.from_pretrained(model).to(device).eval()

    def fn(part):
        texts = [_text(r)[:2000] for _, r in part.iterrows()]
        ids = part["oa_id"].tolist()
        rows = []
        for i in range(0, len(texts), batch_size):
            chunk, cids = texts[i:i+batch_size], ids[i:i+batch_size]
            try:
                preds = m.batch_predict_entities(chunk, labels, threshold=threshold)
            except AttributeError:                      # older gliner
                preds = [m.predict_entities(t, labels, threshold=threshold) for t in chunk]
            for oid, ents in zip(cids, preds):
                for e in ents:
                    rows.append({"oa_id": oid, "source": "gliner",
                                 "entity_type": e["label"], "mention": e["text"],
                                 "concept_id": "", "ontology": "",
                                 "score": round(float(e.get("score", 0)), 3)})
        return pd.DataFrame(rows, columns=ENTITY_COLS)

    return ShardRunner(base, "stage2_gliner", shard_size).run(df, fn)


def run_hunflair2(df, base, shard_size=2000, device=None):
    """HunFlair2: best cross-corpus accuracy (F1 ~60%), normalizes to
    MeSH / NCBI Gene / NCBI Taxonomy / Cellosaurus. 5 types: cell line,
    chemical, disease, gene, species.  !pip install flair"""
    from flair.nn import Classifier
    from flair.data import Sentence
    from flair.tokenization import SciSpacyTokenizer
    import torch
    dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[hunflair2] loading on {dev} (first run downloads ~1-3 GB)")
    tagger = Classifier.load("hunflair2")

    def fn(part):
        rows = []
        for _, r in part.iterrows():
            txt = _text(r)[:5000]
            if not txt:
                continue
            s = Sentence(txt, use_tokenizer=SciSpacyTokenizer())
            tagger.predict(s)
            for lab in s.get_labels():
                rows.append({"oa_id": r["oa_id"], "source": "hunflair2",
                             "entity_type": lab.value.lower(),
                             "mention": lab.data_point.text,
                             "concept_id": "", "ontology": "",
                             "score": round(float(lab.score), 3)})
        return pd.DataFrame(rows, columns=ENTITY_COLS)

    return ShardRunner(base, "stage2_hunflair2", shard_size).run(df, fn)


def run_scispacy(df, base, model="en_ner_bc5cdr_md", linker="mesh",
                 shard_size=10000, n_process=2):
    """scispaCy: CPU, ~30-100 docs/s. Models: en_ner_bc5cdr_md (disease+chemical),
    en_ner_bionlp13cg_md (16 types incl. anatomy/organism), en_ner_craft_md
    (GO/CL/ChEBI/Taxon), en_ner_jnlpba_md (gene/protein).
    Linkers: umls | mesh | rxnorm | go | hpo  (first use downloads ~1 GB).
        !pip install scispacy==0.5.4
        !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
    """
    import spacy
    from scispacy.abbreviation import AbbreviationDetector      # noqa: F401
    from scispacy.linking import EntityLinker                   # noqa: F401
    nlp = spacy.load(model)
    nlp.add_pipe("abbreviation_detector")
    if linker:
        nlp.add_pipe("scispacy_linker",
                     config={"resolve_abbreviations": True, "linker_name": linker})
    onto = {"mesh": "MeSH", "umls": "UMLS", "rxnorm": "RxNorm",
            "go": "GO", "hpo": "HPO"}.get(linker, "")

    def fn(part):
        texts = [_text(r)[:5000] for _, r in part.iterrows()]
        ids = part["oa_id"].tolist()
        rows = []
        for oid, doc in zip(ids, nlp.pipe(texts, batch_size=64, n_process=n_process)):
            for ent in doc.ents:
                cid, sc = "", 0.0
                kb = getattr(ent._, "kb_ents", [])
                if kb:
                    cid, sc = kb[0][0], round(float(kb[0][1]), 3)
                rows.append({"oa_id": oid, "source": f"scispacy:{model}",
                             "entity_type": ent.label_.lower(), "mention": ent.text,
                             "concept_id": cid, "ontology": onto, "score": sc})
        return pd.DataFrame(rows, columns=ENTITY_COLS)

    return ShardRunner(base, f"stage2_scispacy_{model}", shard_size).run(df, fn)


# ============================== STAGE 3: LLM batch (relations + hard cats) ====
LLM_SCHEMA_PROMPT = """You extract structured biomedical information from a scientific abstract.

Return ONLY raw JSON (no markdown, no preamble) with exactly these keys:
{
 "entities": [{"type": "<one of: disease|gene_protein|chemical_drug|species|vector|host|anatomy|cell_type|pathway|phenotype|variant|biomarker|mechanism_of_action|dosage|endpoint|adverse_event>",
               "mention": "<exact span from the text>"}],
 "relations": [{"subject": "<mention>", "subject_type": "<type>",
                "predicate": "<one of: treats|causes|prevents|inhibits|interacts_with|associated_with|transmits|infects|targets>",
                "object": "<mention>", "object_type": "<type>"}]
}
Extract ONLY what the text states. If nothing, return empty lists."""


def llm_batch_build(df, base, model="gpt-4.1-mini", provider="openai",
                    max_abstract=1500, chunk=50000):
    """Write JSONL request files for a Batch API (~50% cheaper, 24h turnaround).
    This is the ONLY practical way to get pathways, dosages, endpoints, adverse
    events, biomarkers, and mechanisms of action — no NER tool covers them."""
    out = _d(base, "stage3_llm")
    files = []
    for ci, start in enumerate(range(0, len(df), chunk)):
        part = df.iloc[start:start + chunk]
        p = out / f"requests_{ci:03d}.jsonl"
        with open(p, "w") as f:
            for _, r in part.iterrows():
                user = (f"TITLE: {str(r.get('oa_title') or '')[:300]}\n"
                        f"ABSTRACT: {str(r.get('oa_abstract') or '')[:max_abstract]}")
                if provider == "openai":
                    req = {"custom_id": str(r["oa_id"]), "method": "POST",
                           "url": "/v1/chat/completions",
                           "body": {"model": model, "temperature": 0, "max_tokens": 900,
                                    "response_format": {"type": "json_object"},
                                    "messages": [{"role": "system", "content": LLM_SCHEMA_PROMPT},
                                                 {"role": "user", "content": user}]}}
                else:  # anthropic message batches
                    req = {"custom_id": str(r["oa_id"]),
                           "params": {"model": model, "max_tokens": 900, "temperature": 0,
                                      "system": LLM_SCHEMA_PROMPT,
                                      "messages": [{"role": "user", "content": user}]}}
                f.write(json.dumps(req) + "\n")
        files.append(str(p))
        print(f"  wrote {p.name}  ({len(part):,} requests)")
    est_in = len(df) * 400 / 1e6
    est_out = len(df) * 300 / 1e6
    print(f"[stage3] {len(df):,} requests | ~{est_in:.1f}M in + ~{est_out:.1f}M out tokens")
    print("  submit (OpenAI):  client.batches.create(input_file_id=..., "
          "endpoint='/v1/chat/completions', completion_window='24h')")
    return files


def llm_batch_submit_openai(path, base):
    """Upload + submit one JSONL to the OpenAI Batch API. Returns batch id."""
    from openai import OpenAI
    c = OpenAI()
    f = c.files.create(file=open(path, "rb"), purpose="batch")
    b = c.batches.create(input_file_id=f.id, endpoint="/v1/chat/completions",
                         completion_window="24h")
    (Path(base) / "stage3_llm" / "batch_ids.txt").open("a").write(f"{b.id}\t{path}\n")
    print(f"  submitted {Path(path).name} -> {b.id} ({b.status})")
    return b.id


def llm_batch_parse(result_paths, base, source="llm"):
    """Parse Batch API result JSONL(s) -> tidy entities + relations."""
    out = _d(base, "stage3_llm")
    ents, rels = [], []
    for rp in ([result_paths] if isinstance(result_paths, str) else result_paths):
        for line in open(rp):
            try:
                rec = json.loads(line)
            except Exception:
                continue
            oid = rec.get("custom_id", "")
            # OpenAI batch shape; Anthropic shape falls through to the fallbacks
            body = (rec.get("response", {}) or {}).get("body", {}) or {}
            txt = ""
            try:
                txt = body["choices"][0]["message"]["content"]
            except Exception:
                try:
                    txt = rec["result"]["message"]["content"][0]["text"]
                except Exception:
                    continue
            txt = re.sub(r"^```(?:json)?|```$", "", str(txt).strip(), flags=re.M).strip()
            m = re.search(r"\{.*\}", txt, re.S)
            if not m:
                continue
            try:
                d = json.loads(m.group(0))
            except Exception:
                continue
            for e in d.get("entities", []) or []:
                ents.append({"oa_id": oid, "source": source,
                             "entity_type": str(e.get("type", "")).lower(),
                             "mention": e.get("mention", ""), "concept_id": "",
                             "ontology": "", "score": 1.0})
            for r in d.get("relations", []) or []:
                rels.append({"oa_id": oid, "source": source,
                             "subj_id": r.get("subject", ""),
                             "subj_type": str(r.get("subject_type", "")).lower(),
                             "predicate": str(r.get("predicate", "")).lower(),
                             "obj_id": r.get("object", ""),
                             "obj_type": str(r.get("object_type", "")).lower(),
                             "evidence": ""})
    E = pd.DataFrame(ents, columns=ENTITY_COLS)
    R = pd.DataFrame(rels, columns=RELATION_COLS)
    E.to_parquet(out / "entities.parquet", index=False)
    R.to_parquet(out / "relations.parquet", index=False)
    print(f"[stage3] {len(E):,} entities | {len(R):,} relations from LLM")
    return E, R


# ============================================ STAGE 4: normalize + combine ====
def crosswalk(entities, base, target="MONDO", source_ontology="MeSH",
              min_score=0.8, chunk=5000):
    """Map mention strings to a target ontology (MONDO/DO/UBERON/GO/HPO/CHEBI)
    with text2term. Use for GLiNER/LLM spans, which carry NO IDs.
        !pip install text2term
    """
    import text2term
    todo = entities.loc[entities["concept_id"].eq("") | entities["concept_id"].isna(),
                        "mention"].dropna().unique().tolist()
    print(f"[stage4] mapping {len(todo):,} unique mentions -> {target}")
    frames = []
    for i in range(0, len(todo), chunk):
        m = text2term.map_terms(todo[i:i+chunk], target, min_score=min_score,
                                mapper=text2term.Mapper.TFIDF)
        frames.append(m)
    mapped = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    mapped.to_parquet(_d(base, "stage4") / f"crosswalk_{target}.parquet", index=False)
    return mapped


def combine(base, out_name="bio"):
    """Union every stage into ONE entities table and ONE relations table."""
    ent_paths = (glob.glob(str(Path(base) / "stage*" / "entities.parquet"))
                 + glob.glob(str(Path(base) / "stage2_*" / "shard_*.parquet")))
    rel_paths = glob.glob(str(Path(base) / "stage*" / "relations.parquet"))

    E = (pd.concat([pd.read_parquet(p) for p in ent_paths], ignore_index=True)
         if ent_paths else pd.DataFrame(columns=ENTITY_COLS))
    R = (pd.concat([pd.read_parquet(p) for p in rel_paths], ignore_index=True)
         if rel_paths else pd.DataFrame(columns=RELATION_COLS))
    E = E.drop_duplicates(["oa_id", "source", "entity_type", "mention"])
    R = R.drop_duplicates(["oa_id", "source", "subj_id", "predicate", "obj_id"])
    d = _d(base, "final")
    E.to_parquet(d / f"{out_name}_entities.parquet", index=False)
    R.to_parquet(d / f"{out_name}_relations.parquet", index=False)

    print(f"\n=== FINAL ===\nentities : {len(E):,}  ({E.oa_id.nunique():,} docs)")
    if len(E):
        print(E.groupby("source").size().to_string())
        print("\ntop entity types:")
        print(E["entity_type"].value_counts().head(10).to_string())
    print(f"\nrelations: {len(R):,}  ({R.oa_id.nunique():,} docs)")
    if len(R):
        print(R["predicate"].value_counts().head(10).to_string())
    return E, R

In [ ]:

        !pip install scispacy==0.5.4
        !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

In [ ]:
!pip install text2term

In [ ]:
!pip install gliner


In [ ]:
%%writefile biomining.py
"""
biomining.py — Biomedical NER / Entity Linking / Relation Extraction at scale
============================================================================
Staged pipeline for ~300k OpenAlex abstracts in Google Colab. Every stage
writes TIDY parquet shards and is resumable — a Colab disconnect costs you at
most one shard.

STAGES (run in this order; each is independent and skippable)
------------------------------------------------------------
  Stage 0  recover_pmids()        DOI -> PMID via NCBI ID Converter (shrinks the
                                  no-PMID set BEFORE you pay for inference)
           fetch_pubtator_bulk()  download PubTator3 bulk TSVs (once)
           join_pubtator()        entities + relations for PMID records, FREE
                                  (no inference: genes, diseases, chemicals,
                                   species, cell lines, variants, 12 relations)
  Stage 1  mesh_from_openalex()   turn the oa_mesh you ALREADY have into tidy
                                  disease/chemical entities — zero cost
  Stage 2  run_scispacy()         CPU spans + MeSH/RxNorm/GO/HPO linking
           run_hunflair2()        GPU NER + normalization (best accuracy)
           run_gliner()           GPU zero-shot NER with YOUR labels
                                  (tick species, vector, host, acaricide, ...)
  Stage 3  llm_batch_build()      JSONL for OpenAI/Anthropic batch APIs
           llm_batch_parse()      -> relations + pathways / dosages / endpoints /
                                     adverse events / biomarkers / MoA
                                  (the categories NO off-the-shelf tool covers)
  Stage 4  crosswalk()            MeSH->MONDO, NCBI Gene->HGNC, etc.
           combine()              one entities table + one relations table

TIDY SCHEMAS (everything joins back on oa_id)
---------------------------------------------
  entities : oa_id, source, entity_type, mention, concept_id, ontology, score
  relations: oa_id, source, subj_id, subj_type, predicate, obj_id, obj_type, evidence

Colab quickstart
----------------
    from google.colab import drive; drive.mount('/content/drive')
    !pip -q install duckdb pandas pyarrow requests
    import biomining as bm
    BASE = "/content/drive/MyDrive/ticks/bio"
    df = pd.read_parquet(".../canonical_screened.parquet")
    df = df[df.oa_relevance_tier.isin(["relevant","screen"])]      # screen first!

    df   = bm.recover_pmids(df, base=BASE)                          # Stage 0
    bm.fetch_pubtator_bulk(base=BASE)                               #  (~6 GB, once)
    bm.join_pubtator(df, base=BASE)
    bm.mesh_from_openalex(df, base=BASE)                            # Stage 1
    todo = bm.no_pubtator_subset(df, base=BASE)                     # the ~200k
    bm.run_gliner(todo, base=BASE)                                  # Stage 2 (GPU)
    bm.llm_batch_build(todo, base=BASE)                             # Stage 3
    ents, rels = bm.combine(base=BASE)                              # Stage 4
"""

from __future__ import annotations
import os, re, io, json, time, glob, gzip, math
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------------- schemas ---
ENTITY_COLS   = ["oa_id","source","entity_type","mention","concept_id","ontology","score"]
RELATION_COLS = ["oa_id","source","subj_id","subj_type","predicate","obj_id","obj_type","evidence"]

# PubTator3 bulk files (https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/)
PUBTATOR_FTP = "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/"
PUBTATOR_FILES = {   # name -> (filename, approx size)
    "gene":     ("gene2pubtator3.gz",      "716 MB"),
    "disease":  ("disease2pubtator3.gz",   "2.0 GB"),
    "chemical": ("chemical2pubtator3.gz",  "1.7 GB"),
    "species":  ("species2pubtator3.gz",   "461 MB"),
    "cellline": ("cellline2pubtator3.gz",  "68 MB"),
    "mutation": ("mutation2pubtator3.gz",  "110 MB"),
    "relation": ("relation2pubtator3.gz",  "277 MB"),
}
# entity_type -> the ontology PubTator3 normalizes to
PUBTATOR_ONTOLOGY = {
    "gene": "NCBI Gene", "disease": "MeSH", "chemical": "MeSH",
    "species": "NCBI Taxonomy", "cellline": "Cellosaurus", "mutation": "dbSNP/HGVS",
}

# Custom labels for GLiNER — the domain-specific stuff fixed-schema tools miss.
TICK_LABELS = [
    "tick species", "tick genus", "pathogen", "disease", "vector", "host animal",
    "reservoir host", "acaricide", "chemical or drug", "gene or protein",
    "biomarker", "geographic location", "control method", "diagnostic method",
]


def _d(base, *p):
    q = Path(base).joinpath(*p); q.mkdir(parents=True, exist_ok=True); return q


# ============================================================ shard runner ===
class ShardRunner:
    """Run `fn(shard_df) -> tidy DataFrame` over a corpus in resumable shards.

    Writes one parquet per shard into <base>/<stage>/. On re-run, shards already
    on disk are skipped — so a Colab disconnect costs you at most one shard.
    """

    def __init__(self, base, stage, shard_size=5000, id_col="oa_id"):
        self.dir = _d(base, stage)
        self.shard_size = shard_size
        self.id_col = id_col
        self.stage = stage

    def done_shards(self):
        return {int(re.search(r"shard_(\d+)", p).group(1))
                for p in glob.glob(str(self.dir / "shard_*.parquet"))}

    def run(self, df, fn, verbose=True):
        n = len(df)
        n_shards = math.ceil(n / self.shard_size)
        done = self.done_shards()
        if verbose:
            print(f"[{self.stage}] {n:,} rows -> {n_shards} shards "
                  f"({len(done)} already done)")
        for i in range(n_shards):
            if i in done:
                continue
            part = df.iloc[i * self.shard_size:(i + 1) * self.shard_size]
            t0 = time.time()
            out = fn(part)
            if out is None or len(out) == 0:
                out = pd.DataFrame(columns=ENTITY_COLS)
            out.to_parquet(self.dir / f"shard_{i:05d}.parquet", index=False)
            if verbose:
                rate = len(part) / max(time.time() - t0, 1e-9)
                eta = (n_shards - i - 1) * self.shard_size / max(rate, 1e-9) / 60
                print(f"  shard {i+1}/{n_shards}  {len(out):,} rows  "
                      f"{rate:.1f} docs/s  ETA {eta:.0f} min")
        return self.load()

    def load(self):
        parts = sorted(glob.glob(str(self.dir / "shard_*.parquet")))
        if not parts:
            return pd.DataFrame(columns=ENTITY_COLS)
        return pd.concat((pd.read_parquet(p) for p in parts), ignore_index=True)


# =================================================== STAGE 0: PMID + PubTator =
def recover_pmids(df, base, id_col="oa_id", batch=200, sleep=0.4):
    """DOI -> PMID via NCBI ID Converter. Run this FIRST: every PMID you recover
    is one abstract you do NOT have to run inference on (PubTator3 has it free).
    Adds/fills `oa_pmid`. Cached to <base>/pmid_map.parquet."""
    import requests
    cache = Path(base) / "pmid_map.parquet"
    known = pd.read_parquet(cache) if cache.exists() else pd.DataFrame(columns=["oa_doi", "pmid"])

    have = df["oa_pmid"].astype(str).str.strip().replace({"nan": "", "None": ""})
    need = df.loc[have.eq("") & df["oa_doi"].astype(str).str.startswith("10."), "oa_doi"]
    need = need[~need.isin(known["oa_doi"])].dropna().unique().tolist()
    print(f"[stage0] have PMID: {(~have.eq('')).sum():,} | resolving {len(need):,} DOIs")

    rows = []
    url = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    for i in range(0, len(need), batch):
        chunk = need[i:i + batch]
        try:
            r = requests.get(url, params={"ids": ",".join(chunk), "format": "json",
                                          "tool": "biomining", "email": "user@example.org"},
                             timeout=60)
            for rec in r.json().get("records", []):
                if rec.get("pmid"):
                    rows.append({"oa_doi": rec.get("doi", ""), "pmid": rec["pmid"]})
        except Exception as e:
            print(f"  chunk {i}: {e}")
        time.sleep(sleep)
        if (i // batch) % 25 == 0 and rows:
            pd.concat([known, pd.DataFrame(rows)]).drop_duplicates("oa_doi").to_parquet(cache, index=False)

    known = pd.concat([known, pd.DataFrame(rows)]).drop_duplicates("oa_doi")
    known.to_parquet(cache, index=False)

    out = df.merge(known, on="oa_doi", how="left")
    out["oa_pmid"] = out["oa_pmid"].astype(str).str.strip().replace({"nan": "", "None": ""})
    out.loc[out["oa_pmid"].eq(""), "oa_pmid"] = out.loc[out["oa_pmid"].eq(""), "pmid"]
    out = out.drop(columns=["pmid"])
    got = out["oa_pmid"].astype(str).ne("").sum()
    print(f"[stage0] PMID coverage now {got:,}/{len(out):,} ({got/len(out):.1%})")
    return out


def fetch_pubtator_bulk(base, which=("disease", "chemical", "gene", "species",
                                     "cellline", "mutation", "relation")):
    """Download PubTator3 bulk TSVs (~5-6 GB total). One-time; cached on Drive."""
    import requests
    d = _d(base, "pubtator_bulk")
    for key in which:
        fn, size = PUBTATOR_FILES[key]
        dest = d / fn
        if dest.exists():
            print(f"  {fn} already present"); continue
        print(f"  downloading {fn} (~{size}) ...")
        with requests.get(PUBTATOR_FTP + fn, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(dest, "wb") as f:
                for c in r.iter_content(chunk_size=1 << 20):
                    f.write(c)
        print(f"    done: {dest.stat().st_size/1e6:.0f} MB")
    return d


def join_pubtator(df, base, which=("disease", "chemical", "gene", "species",
                                   "cellline", "mutation", "relation")):
    """Stream the bulk TSVs with DuckDB, keep only OUR PMIDs, emit tidy
    entities + relations. This is the free lunch: no inference at all."""
    import duckdb
    bulk = Path(base) / "pubtator_bulk"
    out = _d(base, "stage0_pubtator")

    pm = df[["oa_id", "oa_pmid"]].copy()
    pm["oa_pmid"] = pm["oa_pmid"].astype(str).str.strip()
    pm = pm[pm["oa_pmid"].ne("") & pm["oa_pmid"].str.isdigit()]
    if not len(pm):
        print("[stage0] no PMIDs — skipping"); return None, None
    print(f"[stage0] joining PubTator3 for {len(pm):,} PMID records")

    con = duckdb.connect()
    con.register("pm", pm)
    ents, rels = [], []
    for key in which:
        f = bulk / PUBTATOR_FILES[key][0]
        if not f.exists():
            print(f"  (missing {f.name} — skip)"); continue
        if key == "relation":
            # relation2pubtator3: PMID, relation_type, concept_1, concept_2
            q = f"""
                SELECT p.oa_id, 'pubtator3' AS source,
                       CAST(t.column1 AS VARCHAR) AS predicate,
                       CAST(t.column2 AS VARCHAR) AS subj_id,
                       CAST(t.column3 AS VARCHAR) AS obj_id
                FROM read_csv('{f}', delim='\t', header=false, quote='',
                              ignore_errors=true, all_varchar=true) t
                JOIN pm p ON p.oa_pmid = CAST(t.column0 AS VARCHAR)
            """
            r = con.execute(q).df()
            if len(r):
                # relation file encodes concepts as "Type|ID" (e.g. "Species|6945")
                sp = r["subj_id"].str.split("|", n=1)
                op = r["obj_id"].str.split("|", n=1)
                r["subj_type"] = sp.str[0]
                r["subj_id"] = sp.str[-1]
                r["obj_type"] = op.str[0]
                r["obj_id"] = op.str[-1]
                r["evidence"] = ""
                rels.append(r[RELATION_COLS])
        else:
            # <type>2pubtator3: PMID, Type, ConceptID, Mentions, Resource
            q = f"""
                SELECT p.oa_id, 'pubtator3' AS source,
                       lower(CAST(t.column1 AS VARCHAR)) AS entity_type,
                       CAST(t.column3 AS VARCHAR) AS mention,
                       CAST(t.column2 AS VARCHAR) AS concept_id
                FROM read_csv('{f}', delim='\t', header=false, quote='',
                              ignore_errors=true, all_varchar=true) t
                JOIN pm p ON p.oa_pmid = CAST(t.column0 AS VARCHAR)
            """
            e = con.execute(q).df()
            if len(e):
                e["ontology"] = PUBTATOR_ONTOLOGY.get(key, "")
                e["score"] = 1.0
                ents.append(e[ENTITY_COLS])
        print(f"  {key}: {len(ents[-1]) if (key!='relation' and ents) else (len(rels[-1]) if rels and key=='relation' else 0):,} rows")

    E = pd.concat(ents, ignore_index=True) if ents else pd.DataFrame(columns=ENTITY_COLS)
    R = pd.concat(rels, ignore_index=True) if rels else pd.DataFrame(columns=RELATION_COLS)
    E.to_parquet(out / "entities.parquet", index=False)
    R.to_parquet(out / "relations.parquet", index=False)
    print(f"[stage0] {len(E):,} entities | {len(R):,} relations "
          f"| {E.oa_id.nunique():,} docs covered")
    return E, R


def no_pubtator_subset(df, base):
    """The records PubTator3 could NOT cover -> these are what you run NER on."""
    p = Path(base) / "stage0_pubtator" / "entities.parquet"
    covered = set(pd.read_parquet(p)["oa_id"]) if p.exists() else set()
    todo = df[~df["oa_id"].isin(covered)].copy()
    print(f"[stage2] PubTator covered {len(covered):,}; "
          f"{len(todo):,} records need local NER")
    return todo


# ==================================================== STAGE 1: free MeSH ======
def mesh_from_openalex(df, base, sep=" | "):
    """You ALREADY have MeSH for ~1/3 of records (oa_mesh). Turn it into tidy
    entities for free — this substitutes for disease/chemical NER on those docs."""
    out = _d(base, "stage1_mesh")
    s = df[["oa_id", "oa_mesh"]].copy()
    s["oa_mesh"] = s["oa_mesh"].astype("string").fillna("")
    s = s[s["oa_mesh"].ne("")]
    s["mention"] = s["oa_mesh"].str.split(re.escape(sep).replace("\\ ", " "))
    e = s.explode("mention")
    e["mention"] = e["mention"].str.strip()
    e = e[e["mention"].ne("")]
    e["source"] = "openalex_mesh"
    e["entity_type"] = "mesh_descriptor"
    e["concept_id"] = ""          # OpenAlex gives descriptor names, not UIs
    e["ontology"] = "MeSH"
    e["score"] = 1.0
    e = e[ENTITY_COLS]
    e.to_parquet(out / "entities.parquet", index=False)
    print(f"[stage1] {len(e):,} MeSH entities from {e.oa_id.nunique():,} docs (free)")
    return e


# ================================================ STAGE 2: local NER (GPU) ====
def _text(row):
    t = str(row.get("oa_title") or "")
    a = str(row.get("oa_abstract") or "")
    k = str(row.get("oa_abstract") or "")
    return (t + ". " + a + ". " + k).strip()


def run_gliner(df, base, labels=None, model="Ihor/gliner-biomed-bi-large-v1.0",
               shard_size=5000, batch_size=16, threshold=0.5, device=None):
    """Zero-shot NER with YOUR OWN labels — the only tool that captures
    vector/host/reservoir/acaricide roles that fixed-schema taggers miss.
    NOTE: GLiNER gives typed SPANS, not ontology IDs. Normalize in stage 4.
        !pip install gliner
    """
    from gliner import GLiNER
    import torch
    labels = labels or TICK_LABELS
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[gliner] {model} on {device} | {len(labels)} labels")
    m = GLiNER.from_pretrained(model).to(device).eval()

    def fn(part):
        texts = [_text(r)[:2000] for _, r in part.iterrows()]
        ids = part["oa_id"].tolist()
        rows = []
        for i in range(0, len(texts), batch_size):
            chunk, cids = texts[i:i+batch_size], ids[i:i+batch_size]
            try:
                preds = m.batch_predict_entities(chunk, labels, threshold=threshold)
            except AttributeError:                      # older gliner
                preds = [m.predict_entities(t, labels, threshold=threshold) for t in chunk]
            for oid, ents in zip(cids, preds):
                for e in ents:
                    rows.append({"oa_id": oid, "source": "gliner",
                                 "entity_type": e["label"], "mention": e["text"],
                                 "concept_id": "", "ontology": "",
                                 "score": round(float(e.get("score", 0)), 3)})
        return pd.DataFrame(rows, columns=ENTITY_COLS)

    return ShardRunner(base, "stage2_gliner", shard_size).run(df, fn)


def run_hunflair2(df, base, shard_size=2000, device=None):
    """HunFlair2: best cross-corpus accuracy (F1 ~60%), normalizes to
    MeSH / NCBI Gene / NCBI Taxonomy / Cellosaurus. 5 types: cell line,
    chemical, disease, gene, species.  !pip install flair"""
    from flair.nn import Classifier
    from flair.data import Sentence
    from flair.tokenization import SciSpacyTokenizer
    import torch
    dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[hunflair2] loading on {dev} (first run downloads ~1-3 GB)")
    tagger = Classifier.load("hunflair2")

    def fn(part):
        rows = []
        for _, r in part.iterrows():
            txt = _text(r)[:5000]
            if not txt:
                continue
            s = Sentence(txt, use_tokenizer=SciSpacyTokenizer())
            tagger.predict(s)
            for lab in s.get_labels():
                rows.append({"oa_id": r["oa_id"], "source": "hunflair2",
                             "entity_type": lab.value.lower(),
                             "mention": lab.data_point.text,
                             "concept_id": "", "ontology": "",
                             "score": round(float(lab.score), 3)})
        return pd.DataFrame(rows, columns=ENTITY_COLS)

    return ShardRunner(base, "stage2_hunflair2", shard_size).run(df, fn)


def run_scispacy(df, base, model="en_ner_bc5cdr_md", linker="mesh",
                 shard_size=10000, n_process=2):
    """scispaCy: CPU, ~30-100 docs/s. Models: en_ner_bc5cdr_md (disease+chemical),
    en_ner_bionlp13cg_md (16 types incl. anatomy/organism), en_ner_craft_md
    (GO/CL/ChEBI/Taxon), en_ner_jnlpba_md (gene/protein).
    Linkers: umls | mesh | rxnorm | go | hpo  (first use downloads ~1 GB).
        !pip install scispacy==0.5.4
        !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
    """
    import spacy
    from scispacy.abbreviation import AbbreviationDetector      # noqa: F401
    from scispacy.linking import EntityLinker                   # noqa: F401
    nlp = spacy.load(model)
    nlp.add_pipe("abbreviation_detector")
    if linker:
        nlp.add_pipe("scispacy_linker",
                     config={"resolve_abbreviations": True, "linker_name": linker})
    onto = {"mesh": "MeSH", "umls": "UMLS", "rxnorm": "RxNorm",
            "go": "GO", "hpo": "HPO"}.get(linker, "")

    def fn(part):
        texts = [_text(r)[:5000] for _, r in part.iterrows()]
        ids = part["oa_id"].tolist()
        rows = []
        for oid, doc in zip(ids, nlp.pipe(texts, batch_size=64, n_process=n_process)):
            for ent in doc.ents:
                cid, sc = "", 0.0
                kb = getattr(ent._, "kb_ents", [])
                if kb:
                    cid, sc = kb[0][0], round(float(kb[0][1]), 3)
                rows.append({"oa_id": oid, "source": f"scispacy:{model}",
                             "entity_type": ent.label_.lower(), "mention": ent.text,
                             "concept_id": cid, "ontology": onto, "score": sc})
        return pd.DataFrame(rows, columns=ENTITY_COLS)

    return ShardRunner(base, f"stage2_scispacy_{model}", shard_size).run(df, fn)


# ============================== STAGE 3: LLM batch (relations + hard cats) ====
LLM_SCHEMA_PROMPT = """You extract structured biomedical information from a scientific abstract.

Return ONLY raw JSON (no markdown, no preamble) with exactly these keys:
{
 "entities": [{"type": "<one of: disease|gene_protein|chemical_drug|species|vector|host|anatomy|cell_type|pathway|phenotype|variant|biomarker|mechanism_of_action|dosage|endpoint|adverse_event>",
               "mention": "<exact span from the text>"}],
 "relations": [{"subject": "<mention>", "subject_type": "<type>",
                "predicate": "<one of: treats|causes|prevents|inhibits|interacts_with|associated_with|transmits|infects|targets>",
                "object": "<mention>", "object_type": "<type>"}]
}
Extract ONLY what the text states. If nothing, return empty lists."""


def llm_batch_build(df, base, model="gpt-4.1-mini", provider="openai",
                    max_abstract=1500, chunk=50000):
    """Write JSONL request files for a Batch API (~50% cheaper, 24h turnaround).
    This is the ONLY practical way to get pathways, dosages, endpoints, adverse
    events, biomarkers, and mechanisms of action — no NER tool covers them."""
    out = _d(base, "stage3_llm")
    files = []
    for ci, start in enumerate(range(0, len(df), chunk)):
        part = df.iloc[start:start + chunk]
        p = out / f"requests_{ci:03d}.jsonl"
        with open(p, "w") as f:
            for _, r in part.iterrows():
                user = (f"TITLE: {str(r.get('oa_title') or '')[:300]}\n"
                        f"ABSTRACT: {str(r.get('oa_abstract') or '')[:max_abstract]}")
                if provider == "openai":
                    req = {"custom_id": str(r["oa_id"]), "method": "POST",
                           "url": "/v1/chat/completions",
                           "body": {"model": model, "temperature": 0, "max_tokens": 900,
                                    "response_format": {"type": "json_object"},
                                    "messages": [{"role": "system", "content": LLM_SCHEMA_PROMPT},
                                                 {"role": "user", "content": user}]}}
                else:  # anthropic message batches
                    req = {"custom_id": str(r["oa_id"]),
                           "params": {"model": model, "max_tokens": 900, "temperature": 0,
                                      "system": LLM_SCHEMA_PROMPT,
                                      "messages": [{"role": "user", "content": user}]}}
                f.write(json.dumps(req) + "\n")
        files.append(str(p))
        print(f"  wrote {p.name}  ({len(part):,} requests)")
    est_in = len(df) * 400 / 1e6
    est_out = len(df) * 300 / 1e6
    print(f"[stage3] {len(df):,} requests | ~{est_in:.1f}M in + ~{est_out:.1f}M out tokens")
    print("  submit (OpenAI):  client.batches.create(input_file_id=..., "
          "endpoint='/v1/chat/completions', completion_window='24h')")
    return files


def llm_batch_submit_openai(path, base):
    """Upload + submit one JSONL to the OpenAI Batch API. Returns batch id."""
    from openai import OpenAI
    c = OpenAI()
    f = c.files.create(file=open(path, "rb"), purpose="batch")
    b = c.batches.create(input_file_id=f.id, endpoint="/v1/chat/completions",
                         completion_window="24h")
    (Path(base) / "stage3_llm" / "batch_ids.txt").open("a").write(f"{b.id}\t{path}\n")
    print(f"  submitted {Path(path).name} -> {b.id} ({b.status})")
    return b.id


def llm_batch_parse(result_paths, base, source="llm"):
    """Parse Batch API result JSONL(s) -> tidy entities + relations."""
    out = _d(base, "stage3_llm")
    ents, rels = [], []
    for rp in ([result_paths] if isinstance(result_paths, str) else result_paths):
        for line in open(rp):
            try:
                rec = json.loads(line)
            except Exception:
                continue
            oid = rec.get("custom_id", "")
            # OpenAI batch shape; Anthropic shape falls through to the fallbacks
            body = (rec.get("response", {}) or {}).get("body", {}) or {}
            txt = ""
            try:
                txt = body["choices"][0]["message"]["content"]
            except Exception:
                try:
                    txt = rec["result"]["message"]["content"][0]["text"]
                except Exception:
                    continue
            txt = re.sub(r"^```(?:json)?|```$", "", str(txt).strip(), flags=re.M).strip()
            m = re.search(r"\{.*\}", txt, re.S)
            if not m:
                continue
            try:
                d = json.loads(m.group(0))
            except Exception:
                continue
            for e in d.get("entities", []) or []:
                ents.append({"oa_id": oid, "source": source,
                             "entity_type": str(e.get("type", "")).lower(),
                             "mention": e.get("mention", ""), "concept_id": "",
                             "ontology": "", "score": 1.0})
            for r in d.get("relations", []) or []:
                rels.append({"oa_id": oid, "source": source,
                             "subj_id": r.get("subject", ""),
                             "subj_type": str(r.get("subject_type", "")).lower(),
                             "predicate": str(r.get("predicate", "")).lower(),
                             "obj_id": r.get("object", ""),
                             "obj_type": str(r.get("object_type", "")).lower(),
                             "evidence": ""})
    E = pd.DataFrame(ents, columns=ENTITY_COLS)
    R = pd.DataFrame(rels, columns=RELATION_COLS)
    E.to_parquet(out / "entities.parquet", index=False)
    R.to_parquet(out / "relations.parquet", index=False)
    print(f"[stage3] {len(E):,} entities | {len(R):,} relations from LLM")
    return E, R


# ============================================ STAGE 4: normalize + combine ====
def crosswalk(entities, base, target="MONDO", source_ontology="MeSH",
              min_score=0.8, chunk=5000):
    """Map mention strings to a target ontology (MONDO/DO/UBERON/GO/HPO/CHEBI)
    with text2term. Use for GLiNER/LLM spans, which carry NO IDs.
        !pip install text2term
    """
    import text2term
    todo = entities.loc[entities["concept_id"].eq("") | entities["concept_id"].isna(),
                        "mention"].dropna().unique().tolist()
    print(f"[stage4] mapping {len(todo):,} unique mentions -> {target}")
    frames = []
    for i in range(0, len(todo), chunk):
        m = text2term.map_terms(todo[i:i+chunk], target, min_score=min_score,
                                mapper=text2term.Mapper.TFIDF)
        frames.append(m)
    mapped = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    mapped.to_parquet(_d(base, "stage4") / f"crosswalk_{target}.parquet", index=False)
    return mapped


def combine(base, out_name="bio"):
    """Union every stage into ONE entities table and ONE relations table."""
    ent_paths = (glob.glob(str(Path(base) / "stage*" / "entities.parquet"))
                 + glob.glob(str(Path(base) / "stage2_*" / "shard_*.parquet")))
    rel_paths = glob.glob(str(Path(base) / "stage*" / "relations.parquet"))

    E = (pd.concat([pd.read_parquet(p) for p in ent_paths], ignore_index=True)
         if ent_paths else pd.DataFrame(columns=ENTITY_COLS))
    R = (pd.concat([pd.read_parquet(p) for p in rel_paths], ignore_index=True)
         if rel_paths else pd.DataFrame(columns=RELATION_COLS))
    E = E.drop_duplicates(["oa_id", "source", "entity_type", "mention"])
    R = R.drop_duplicates(["oa_id", "source", "subj_id", "predicate", "obj_id"])
    d = _d(base, "final")
    E.to_parquet(d / f"{out_name}_entities.parquet", index=False)
    R.to_parquet(d / f"{out_name}_relations.parquet", index=False)

    print(f"\n=== FINAL ===\nentities : {len(E):,}  ({E.oa_id.nunique():,} docs)")
    if len(E):
        print(E.groupby("source").size().to_string())
        print("\ntop entity types:")
        print(E["entity_type"].value_counts().head(10).to_string())
    print(f"\nrelations: {len(R):,}  ({R.oa_id.nunique():,} docs)")
    if len(R):
        print(R["predicate"].value_counts().head(10).to_string())
    return E, R


In [ ]:
    from google.colab import drive; drive.mount('/content/drive')
    !pip -q install duckdb pandas pyarrow requests
    import biomining as bm
    import pandas as pd

    from pathlib import Path

    BASE = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/bio"
    out = Path(BASE); out.mkdir(parents=True, exist_ok=True)


    df = pd.read_parquet("/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/canonical_screened.parquet")
    df = df[df.oa_relevance_tier.isin(["relevant","screen"])]      # screen first!

    df   = bm.recover_pmids(df, base=BASE)                          # Stage 0
    bm.fetch_pubtator_bulk(base=BASE)                               #  (~6 GB, once)
    bm.join_pubtator(df, base=BASE)
    bm.mesh_from_openalex(df, base=BASE)                            # Stage 1
    todo = bm.no_pubtator_subset(df, base=BASE)                     # the ~200k
    bm.run_gliner(todo, base=BASE)                                  # Stage 2 (GPU)
    bm.llm_batch_build(todo, base=BASE)                             # Stage 3
    ents, rels = bm.combine(base=BASE)                              # Stage 4

In [ ]:
todo

In [ ]:
rels

In [ ]:
ents

In [ ]:
ents.sample(60)